# Phase 1 — Baseline testing: CheXagent-2-3b on chest X-ray VQA

**Goal of this notebook:** run the pretrained, open-source `StanfordAIMI/CheXagent-2-3b`
model zero-shot (no training, no retrieval, no modification) on a small chest-X-ray
question set, and check whether it shows the same weak spot its own paper reports:
strong on general/coarse questions (presence, modality, organ), weaker on fine-grained
**laterality** ("which side") and **severity** ("mild vs. severe") questions.

This is Phase 1 of 3:
1. **Baseline testing** (this notebook) — measure where the off-the-shelf model is weak.
2. **RAG layer** — add knowledge-grounded retrieval targeted at the weak spot(s) found here.
3. **Fine-tuning (if needed)** — only if RAG alone doesn't close the gap.

**Important caveat, stated up front, not hidden:** CheXagent's own training corpus
(CheXinstruct) explicitly includes VQA-RAD, SLAKE, and Open-I among its 32 source
datasets. So this is *not* a clean, held-out, zero-shot generalization test — it's close
to in-distribution. That's normal for this field (every paper we reviewed reuses the same
public benchmarks for the same reason). The point here isn't to claim a novel accuracy
number; it's to get our *own* measured numbers, on data we control end-to-end, for the
specific failure mode we plan to target with RAG in Phase 2.

**Runtime:** Free Google Colab, **T4 GPU** (Runtime → Change runtime type → T4 GPU).
Everything below — model download, inference, evaluation — fits comfortably in the
free tier's 16GB VRAM and disk quota.


## 1. Setup

CheXagent ships its own model code, written for `transformers==4.40.0`. That exact version
cannot be installed on Colab's current Python (its `tokenizers` dependency has no prebuilt
package there), and versions newer than 4.46 break the model's `generate` code. So we use
**`transformers==4.46.3`** and, in section 3, patch the two hard-coded version checks in a local
copy of the model code. This combination was tested end-to-end (tokenizer, image handling,
model build, `generate`) on a small random-weight copy of the model before shipping.

If Colab asks you to **restart the runtime** after this cell, do so, then continue from the next
cell (do not re-run the install cell).


In [ ]:
#@title Install dependencies (torch is preinstalled on Colab)
!pip install -q "transformers==4.46.3" accelerate einops sentencepiece protobuf pyarrow albumentations huggingface_hub
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")


## 2. Build the chest X-ray test set

Two sources, combined:
- **SLAKE** (test + validation) — has structured `location` + `modality` metadata, so chest
  X-ray rows are filtered precisely.
- **VQA-RAD** — has no organ metadata, so chest rows are selected with a keyword heuristic
  (noisier — flagged here, not hidden; a few non-chest items may still slip through, which is
  why section 6 shows images of wrong answers for you to check).

Every question is tagged by simple keyword rules (the next cell shows them):
- `category`: `laterality` / `severity` / `general`
- `answer_format`: `yes/no` vs `open`
- `is_knowledge`: answered from medical knowledge (symptoms, treatment...) vs by looking at the image

Changes after the first pilot run (60 questions): the radiation-damage template question is
dropped (it is not clinical severity), identical question text is capped at 4 per category, more
questions are sampled, and question IDs are stable so Phase 2 can be compared row by row.

The two cells below are the repo's own `baseline/categorize.py` and `baseline/data.py`,
inlined unchanged (they are unit-tested in the repo).


In [ ]:
"""Heuristic question-type categorization.

The literature review (CheXagent in particular) found that CXR VLMs score
~90-99% on coarse perception questions (presence/absence, modality, view,
organ identification) but drop to ~77-79% on fine-grained laterality and
severity questions ("which side", "mild vs. severe"). To test whether our
baseline shows the same weak spot, every question in the baseline test set
is tagged with a category up front so results can be broken down by category
rather than reported as one aggregate accuracy number.

This is a deliberately simple keyword heuristic, not a trained classifier.
It will mis-tag some edge cases (e.g. "what is to the left of the heart" is
positional, not a laterality-of-finding question) -- that is an accepted,
documented limitation, not a bug to silently paper over.
"""


import re

LATERALITY_PATTERNS = [
    r"\bleft\b",
    r"\bright\b",
    r"\bbilateral\b",
    r"\bwhich side\b",
    r"\blaterality\b",
    r"\bunilateral\b",
]

SEVERITY_PATTERNS = [
    r"\bmild\b",
    r"\bmoderate\b",
    r"\bsevere\b",
    r"\benlarge(d|ment)?\b",
    r"\bdegree\b",
    r"\bextent\b",
    r"\bhow (much|severe|large|big)\b",
    r"\bsize of\b",
    r"\bworsen(ed|ing)?\b",
    r"\bimprov(ed|ing|ement)\b",
    r"\bsmall\b",
    r"\blarge\b",
]

KNOWLEDGE_PATTERNS = [
    r"\bsymptoms?\b",
    r"\bhow to (treat|prevent|cure)\b",
    r"\b(treatment|therapy|prevention)\b",
    r"\bfunction of\b",
    r"\bcause[sd]?\b",
    r"\bhow do you (determine|diagnose|assess|measure)\b",
    # Added with the knowledge test set: 'why ...?' and 'how can you tell ...?'
    # are answered from anatomy/physiology knowledge. Run 2's saved tags were
    # produced before these two were added.
    r"\bwhy\b",
    r"\bhow can you tell\b",
]

# Questions that are templates about the imaging procedure, not about the
# patient's findings (e.g. SLAKE's "How much damage will be caused to the body
# taking such an image?"). They match the severity keywords ("how much") but are
# not clinical severity, so the data pipeline drops them.
NON_CLINICAL_PATTERNS = [
    r"\bdamage\b.*\btaking such an image\b",
    r"\bradiation\b",
]

CATEGORY_LATERALITY = "laterality"
CATEGORY_SEVERITY = "severity"
CATEGORY_GENERAL = "general"

ALL_CATEGORIES = (CATEGORY_LATERALITY, CATEGORY_SEVERITY, CATEGORY_GENERAL)


def _matches_any(text: str, patterns: list[str]) -> bool:
    return any(re.search(p, text) for p in patterns)


def categorize_question(question: str) -> str:
    """Tag a question as 'laterality', 'severity', or 'general'.

    Laterality is checked before severity because a question can contain
    both kinds of words ("is the left-sided effusion severe?") -- in our
    taxonomy laterality takes priority since it is the harder, more
    specifically-flagged failure mode in CheXagent's own reported numbers.
    """
    q = question.lower().strip()
    if _matches_any(q, LATERALITY_PATTERNS):
        return CATEGORY_LATERALITY
    if _matches_any(q, SEVERITY_PATTERNS):
        return CATEGORY_SEVERITY
    return CATEGORY_GENERAL


def is_knowledge_question(question: str) -> bool:
    """True for questions answered mostly from medical knowledge (symptoms,
    treatment, prevention, how a finding is measured) rather than from looking
    at the image. These are what a retrieval layer can plausibly help with;
    perception questions (is X present, which side) are not.

    Independent of `categorize_question`: a question can be both a laterality
    question and a knowledge question ("How to treat the disease on the lower
    left of lung?")."""
    q = question.lower().strip()
    return _matches_any(q, KNOWLEDGE_PATTERNS)


_SUBTYPE_RULES = [
    ("symptoms", r"\bsymptoms?\b"),
    ("treatment", r"\b(how to treat|treatment|therapy)\b"),
    ("prevention", r"\b(prevent|prevention)\b"),
    ("cause", r"\bcause[sd]?\b"),
    ("function", r"\bfunction of\b"),
    ("explanation", r"\b(why|how can you tell|how do you (determine|diagnose|assess|measure))\b"),
]


def knowledge_subtype(question: str) -> str:
    """Which kind of knowledge question this is (symptoms / treatment /
    prevention / cause / function / explanation / other). Reported separately
    because the SLAKE knowledge questions come from a few fixed templates and
    a model can score well on one template and not another."""
    q = question.lower().strip()
    for name, pattern in _SUBTYPE_RULES:
        if re.search(pattern, q):
            return name
    return "other"


def is_non_clinical_question(question: str) -> bool:
    q = question.lower().strip()
    return _matches_any(q, NON_CLINICAL_PATTERNS)


def answer_format(gold_answer: str) -> str:
    """'yes/no' for closed questions, 'open' otherwise. Reported separately
    because a model that mostly answers 'no' looks decent on yes/no questions
    when most true answers are 'no'."""
    g = gold_answer.lower().strip().rstrip(".")
    return "yes/no" if g in ("yes", "no") else "open"


In [ ]:
"""Build a small, chest-X-ray-only VQA test set for Phase 1 baseline testing.

Important caveat (documented here, not hidden): CheXagent's own training
corpus (CheXinstruct) explicitly rolls in VQA-RAD, SLAKE, and Open-I/IU X-ray
among its 32 source datasets. So evaluating CheXagent on these test splits is
*not* a clean held-out/zero-shot measurement -- it is close to in-distribution.
That is simply the state of public CXR VQA benchmarks today (virtually every
other paper in our review reuses the same splits for the same reason).

The point of Phase 1 is not to claim a novel generalization number; it is to
check whether our own baseline run reproduces the *specific, already-known*
weak spot from CheXagent's paper (strong on coarse/general questions, weaker
on laterality/severity) on data we control end-to-end, so Phase 2 (RAG) has a
concrete, measured gap to target rather than a hypothetical one.

Two sources are combined:
- SLAKE (test + validation): has structured `location` + `modality` metadata, so
  chest X-ray rows can be filtered precisely (no guessing from question text).
- VQA-RAD: has no organ metadata, so chest rows are selected with a keyword
  heuristic (noisier, clearly flagged below -- not a silent assumption).

Non-clinical template questions (e.g. SLAKE's radiation-damage question) are
dropped, repeated question text is capped per category, and every record gets a
stable `id` taken from the source dataset's own row id so later runs (RAG) can
be compared with this baseline row by row.
"""


import hashlib
import json
import random
import re
import time
import zipfile
from pathlib import Path

import requests


SLAKE_DATASET = "BoKelvin/SLAKE"
SLAKE_IMAGES_ZIP_URL = "https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip"
VQA_RAD_DATASET = "flaviagiammarino/vqa-rad"
DATASETS_SERVER_ROWS_URL = "https://datasets-server.huggingface.co/rows"

# Preferred data path: download each dataset's own files once (5 requests).
# The paging API (`datasets-server`, ~150 requests) rate-limits Colab with HTTP
# 429, so it is kept only as a fallback. Row ids are identical on both paths,
# checked against all 224 question ids of run 2.
HF_RESOLVE_URL = "https://huggingface.co/datasets/{repo}/resolve/main/{path}"
SLAKE_SPLIT_FILES = {"test": "test.json", "validation": "validation.json", "train": "train.json"}
VQA_RAD_PARQUET_FILES = {
    "test": "data/test-00000-of-00001-e5bc3d208bb4deeb.parquet",
    "train": "data/train-00000-of-00001-eb8844602202be60.parquet",
}
VQA_RAD_PARQUET_KEY_PREFIX = "vqarad-parquet"
RAW_CACHE_DIR = Path("raw_cache")

# SLAKE's `location` field uses these labels for chest X-ray images.
SLAKE_CHEST_LOCATIONS = {"Chest_heart", "Chest_lung", "Chest_mediastinal", "Lung"}

# VQA-RAD keyword heuristic: a question must mention a chest-related term and
# must NOT mention a clearly non-chest body region. This is approximate --
# some true negatives/positives will slip through -- which is why SLAKE
# (precise metadata) is the primary source and VQA-RAD is secondary/noisier.
VQA_RAD_CHEST_KEYWORDS = {
    "lung", "lungs", "chest", "pulmonary", "pleural", "pneumothorax",
    "cardiac", "heart", "mediastin", "rib", "costophrenic", "consolidation",
    "effusion", "atelectasis", "diaphragm", "thora", "cardiomegaly",
    "pneumonia", "hilar", "bronch",
}
# Keywords that must match as a whole word. "rib" is the one that matters: as a
# substring it matched "describe", "circumscribed", "distribution" and
# "tributaries", which let brain/abdominal "describe the lesion" questions into
# the chest set (found by looking at the images of the side set).
VQA_RAD_WHOLE_WORD_KEYWORDS = {"rib", "ribs"}
_WHOLE_WORD_RE = re.compile(r"\b(" + "|".join(sorted(VQA_RAD_WHOLE_WORD_KEYWORDS)) + r")\b")

VQA_RAD_EXCLUDE_KEYWORDS = {
    "brain", "skull", "head", "abdomen", "abdominal", "liver", "kidney",
    "pelvis", "pelvic", "spine", "spinal", "bowel", "intestine", "stomach",
    "bladder", "uterus", "ovary", "appendix", "cerebral", "cranial", "sinus",
    # Added after the pilot run surfaced non-chest items (e.g. "vascular
    # distribution of these infarcts"). Still a heuristic -- review the images.
    "infarct", "ventric", "sulci", "gyri", "white matter", "gray matter",
    "grey matter", "falx", "midline", "mass effect", "vascular distribution",
    "hepatic", "splenic", "spleen", "adrenal", "pancrea", "gallbladder",
    "renal", "psoas", "femur", "knee", "shoulder", "orbit",
}


def mentions_chest_term(question: str) -> bool:
    """True if the question text contains a chest keyword (substring match,
    except for short words listed in VQA_RAD_WHOLE_WORD_KEYWORDS)."""
    q = question.lower()
    if _WHOLE_WORD_RE.search(q):
        return True
    return any(k in q for k in VQA_RAD_CHEST_KEYWORDS if k not in VQA_RAD_WHOLE_WORD_KEYWORDS)


def _retry_wait(resp, attempt: int, backoff: float) -> float:
    """Seconds to wait before retrying. Honours Retry-After on HTTP 429 and
    otherwise backs off exponentially (capped at 60 s)."""
    if resp is not None and resp.status_code == 429:
        try:
            return min(120.0, max(float(resp.headers.get("Retry-After", 0)), backoff * 4 * (attempt + 1)))
        except ValueError:
            pass
    return min(60.0, backoff * (2 ** attempt))


def _download_file(url: str, dest: str | Path, retries: int = 8, backoff: float = 2.0) -> Path:
    """Download `url` to `dest` once (skipped if present), streaming, with
    retries on 429 / 5xx / connection errors. Written via a temp file so an
    interrupted download is never mistaken for a complete one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    last_exc: Exception | None = None
    for attempt in range(retries):
        resp = None
        try:
            resp = requests.get(url, stream=True, timeout=180)
            if resp.status_code == 200:
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                tmp.replace(dest)
                return dest
            if resp.status_code not in (429, 500, 502, 503, 504):
                resp.raise_for_status()
        except requests.RequestException as exc:
            last_exc = exc
            if resp is not None and resp.status_code not in (429, 500, 502, 503, 504):
                raise
        time.sleep(_retry_wait(resp, attempt, backoff))
    raise last_exc or RuntimeError(f"could not download {url}")


def _get_rows(dataset: str, split: str, offset: int, length: int = 100,
              retries: int = 8, backoff: float = 1.5) -> dict:
    last_exc = None
    for attempt in range(retries):
        try:
            resp = requests.get(
                DATASETS_SERVER_ROWS_URL,
                params={
                    "dataset": dataset, "config": "default", "split": split,
                    "offset": offset, "length": length,
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()
        except requests.RequestException as exc:
            last_exc = exc
            resp = None
        time.sleep(_retry_wait(resp, attempt, backoff))
    if last_exc:
        raise last_exc
    resp.raise_for_status()
    return {}


def _iter_all_rows(dataset: str, split: str, page_size: int = 100):
    offset = 0
    while True:
        data = _get_rows(dataset, split, offset, length=page_size)
        rows = data.get("rows", [])
        if not rows:
            return
        for row in rows:
            yield {**row["row"], "_row_idx": row["row_idx"]}
        offset += len(rows)
        if offset >= data.get("num_rows_total", 0):
            return


def _iter_slake_rows(split: str):
    """SLAKE rows from the dataset's own JSON file; paging API as fallback."""
    try:
        url = HF_RESOLVE_URL.format(repo=SLAKE_DATASET, path=SLAKE_SPLIT_FILES[split])
        path = _download_file(url, RAW_CACHE_DIR / f"slake_{split}.json")
        rows = json.loads(path.read_text())
    except Exception as exc:
        print(f"[data] SLAKE {split}: file download failed ({exc!r}); falling back to the paging API")
        yield from _iter_all_rows(SLAKE_DATASET, split)
        return
    yield from rows


def fetch_slake_chest_records(split: str = "test") -> list[dict]:
    """English chest-X-ray rows from SLAKE, filtered via structured metadata."""
    records = []
    for rr in _iter_slake_rows(split):
        if (
            rr.get("modality") == "X-Ray"
            and rr.get("location") in SLAKE_CHEST_LOCATIONS
            and rr.get("q_lang") == "en"
            and not is_non_clinical_question(rr["question"])
        ):
            records.append({
                "source_dataset": "SLAKE",
                "source_split": split,
                "question": rr["question"],
                "gold_answer": rr["answer"],
                "answer_type": rr.get("answer_type"),
                "uid": f"SLAKE_{rr['qid']}",
                "image_key": rr["img_name"],  # e.g. "xmlab102/source.jpg"
                "category": categorize_question(rr["question"]),
                "answer_format": answer_format(rr["answer"]),
                "is_knowledge": is_knowledge_question(rr["question"]),
            })
    return records


def _vqa_rad_parquet_path(split: str) -> Path:
    url = HF_RESOLVE_URL.format(repo=VQA_RAD_DATASET, path=VQA_RAD_PARQUET_FILES[split])
    return _download_file(url, RAW_CACHE_DIR / f"vqa_rad_{split}.parquet")


def _iter_vqa_rad_rows(split: str):
    """VQA-RAD rows from the dataset's parquet file (images are embedded, so
    there are no expiring signed URLs, and an image's content hash gives a real
    image identity); paging API as fallback if pyarrow or the download fails.

    Yields dicts with question, answer, _row_idx, _image_key, _image_id."""
    try:
        import pyarrow.parquet as pq
        table = pq.read_table(_vqa_rad_parquet_path(split))
        images, questions, answers = table.column("image"), table.column("question"), table.column("answer")
        rows = []
        for idx in range(table.num_rows):
            img_bytes = images[idx].as_py()["bytes"]
            rows.append({
                "question": questions[idx].as_py(), "answer": answers[idx].as_py(), "_row_idx": idx,
                "_image_key": f"{VQA_RAD_PARQUET_KEY_PREFIX}:{split}:{idx}",
                "_image_id": hashlib.md5(img_bytes).hexdigest()[:12],
            })
    except Exception as exc:
        print(f"[data] VQA-RAD {split}: parquet path failed ({exc!r}); falling back to the paging API")
        for rr in _iter_all_rows(VQA_RAD_DATASET, split):
            yield {"question": rr["question"], "answer": rr["answer"], "_row_idx": rr["_row_idx"],
                   "_image_key": rr["image"]["src"], "_image_id": None}
        return
    yield from rows


def fetch_vqa_rad_chest_records(splits: tuple[str, ...] = ("test", "train")) -> list[dict]:
    """Chest-ish rows from VQA-RAD, filtered via keyword heuristic (noisier)."""
    records = []
    for split in splits:
        for rr in _iter_vqa_rad_rows(split):
            q = rr["question"].lower()
            if (
                mentions_chest_term(q)
                and not any(k in q for k in VQA_RAD_EXCLUDE_KEYWORDS)
                and not is_non_clinical_question(rr["question"])
            ):
                records.append({
                    "source_dataset": "VQA-RAD",
                    "source_split": split,
                    "question": rr["question"],
                    "gold_answer": rr["answer"],
                    "answer_type": None,
                    "uid": f"VQA-RAD_{split}_{rr['_row_idx']}",
                    "image_key": rr["_image_key"],
                    "image_id": rr["_image_id"],  # content hash of the image (None on the fallback path)
                    "category": categorize_question(rr["question"]),
                    "answer_format": answer_format(rr["answer"]),
                    "is_knowledge": is_knowledge_question(rr["question"]),
                })
    return records


def stratified_sample(records: list[dict], n_per_category: int, seed: int = 0,
                      max_per_question: int = 4) -> list[dict]:
    """Sample up to `n_per_category` records per category.

    `max_per_question` caps how many rows with identical question text can
    be drawn (the pilot run had "is the heart enlarged?" ~12 times in 60 rows,
    which lets one question dominate a category). Deterministic for a given
    input order and seed.
    """
    rng = random.Random(seed)
    by_cat: dict[str, list[dict]] = {}
    for r in records:
        by_cat.setdefault(r["category"], []).append(r)
    sampled = []
    for cat in sorted(by_cat):
        shuffled = by_cat[cat][:]
        rng.shuffle(shuffled)
        per_question: dict[str, int] = {}
        taken = []
        for r in shuffled:
            key = r["question"].lower().strip()
            if per_question.get(key, 0) >= max_per_question:
                continue
            per_question[key] = per_question.get(key, 0) + 1
            taken.append(r)
            if len(taken) >= n_per_category:
                break
        sampled.extend(taken)
    rng.shuffle(sampled)
    return sampled


def pool_summary(records: list[dict]) -> dict[str, dict[str, int]]:
    """Counts per category / source / answer format, so it is visible how many
    questions of each kind actually exist (severity questions are rare in
    public CXR VQA sets -- that is itself a finding)."""
    summary: dict[str, dict[str, int]] = {"category": {}, "source": {}, "answer_format": {}}
    for r in records:
        for field, key in (("category", r["category"]), ("source", r["source_dataset"]),
                           ("answer_format", r["answer_format"])):
            summary[field][key] = summary[field].get(key, 0) + 1
    return summary


def download_slake_images(records: list[dict], dest_dir: str | Path,
                           zip_cache_path: str | Path | None = None) -> dict[str, str]:
    """Download SLAKE's imgs.zip once (cached) and extract only needed files."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    needed_keys = {r["image_key"] for r in records if r["source_dataset"] == "SLAKE"}
    if not needed_keys:
        return {}

    zip_cache_path = Path(zip_cache_path) if zip_cache_path else dest_dir / "_slake_imgs.zip"
    _download_file(SLAKE_IMAGES_ZIP_URL, zip_cache_path)

    local_paths = {}
    with zipfile.ZipFile(zip_cache_path) as zf:
        for key in needed_keys:
            member = f"imgs/{key}"
            out_path = dest_dir / key.replace("/", "__")
            if not out_path.exists():
                with zf.open(member) as src, open(out_path, "wb") as dst:
                    dst.write(src.read())
            local_paths[key] = str(out_path)
    return local_paths


def download_vqa_rad_images(records: list[dict], dest_dir: str | Path) -> dict[str, str]:
    """Write each needed VQA-RAD image to `dest_dir`. Records from the parquet
    path are extracted from the parquet file and named by image content hash, so
    questions about the same image share one file. Records from the paging
    fallback are fetched from their (expiring) signed URLs."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    local_paths: dict[str, str] = {}
    tables: dict[str, object] = {}
    for r in records:
        if r["source_dataset"] != "VQA-RAD":
            continue
        key = r["image_key"]
        if key in local_paths:
            continue
        if key.startswith(VQA_RAD_PARQUET_KEY_PREFIX + ":"):
            _, split, idx = key.split(":")
            out_path = dest_dir / f"vqarad_{r['image_id']}.jpg"
            if not out_path.exists():
                if split not in tables:
                    import pyarrow.parquet as pq
                    tables[split] = pq.read_table(_vqa_rad_parquet_path(split), columns=["image"])
                out_path.write_bytes(tables[split].column("image")[int(idx)].as_py()["bytes"])
        else:
            digest = hashlib.md5(key.encode()).hexdigest()[:12]
            out_path = dest_dir / f"vqarad_{digest}.jpg"
            if not out_path.exists():
                resp = requests.get(key, timeout=60)
                resp.raise_for_status()
                with open(out_path, "wb") as f:
                    f.write(resp.content)
        local_paths[key] = str(out_path)
    return local_paths


def build_baseline_test_set(images_dir: str | Path, n_per_category: int = 100,
                             seed: int = 0, max_per_question: int = 4,
                             verbose: bool = True) -> list[dict]:
    """End-to-end: fetch metadata, stratify-sample per category, download
    images, return records ready for model inference. Each record has an
    `image_path` and a stable `id` (derived from the source dataset's own row
    id, so baseline and later RAG runs can be compared row by row)."""
    slake = (fetch_slake_chest_records(split="test")
             + fetch_slake_chest_records(split="validation"))
    vqa_rad = fetch_vqa_rad_chest_records(splits=("test", "train"))
    all_records = slake + vqa_rad
    if verbose:
        print("Available pool before sampling:", pool_summary(all_records))

    sampled = stratified_sample(all_records, n_per_category=n_per_category, seed=seed,
                                max_per_question=max_per_question)

    slake_records = [r for r in sampled if r["source_dataset"] == "SLAKE"]
    vqa_rad_records = [r for r in sampled if r["source_dataset"] == "VQA-RAD"]
    slake_paths = download_slake_images(slake_records, images_dir)
    vqa_rad_paths = download_vqa_rad_images(vqa_rad_records, images_dir)

    for r in sampled:
        paths = slake_paths if r["source_dataset"] == "SLAKE" else vqa_rad_paths
        r["image_path"] = paths[r["image_key"]]
        r["id"] = r["uid"]

    return sampled


In [ ]:
# Up to N_PER_CATEGORY questions per category (general / laterality / severity).
# Severity questions are rare in public CXR VQA sets, so that bucket may come out smaller than
# the others -- the printed pool summary shows exactly how many exist.
N_PER_CATEGORY = 100

test_set = build_baseline_test_set(images_dir="images", n_per_category=N_PER_CATEGORY, seed=0)

from collections import Counter
print(f"Built {len(test_set)} questions")
print("By category:     ", Counter(r["category"] for r in test_set))
print("By answer format:", Counter(r["answer_format"] for r in test_set))
print("By source:       ", Counter(r["source_dataset"] for r in test_set))
print("Knowledge-type:  ", sum(r["is_knowledge"] for r in test_set))


## 3. Load CheXagent-2-3b

First run downloads the model from Hugging Face (not gated, MIT license, no token needed).
The repo stores the weights in float32, so the **download is ~12.6GB** (a few minutes; Colab has
the disk space). It is then loaded in half precision (`float16` on a T4, which has no native
`bfloat16`; `bfloat16` on newer GPUs), which fits a free T4's 16GB VRAM.

We download to a local folder first so we can patch the two `assert transformers.__version__ ==
"4.40.0"` lines in the model's own code (see section 1).


In [ ]:
import glob, re
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "StanfordAIMI/CheXagent-2-3b"
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Using dtype:", DTYPE)

_VERSION_ASSERT = re.compile(r'^assert transformers\.__version__ == "4\.40\.0".*$', flags=re.M)

def patch_version_asserts(model_dir):
    """Neutralize the model code's hard-coded transformers==4.40.0 checks.
    Returns the number of lines patched (expected: 2)."""
    patched = 0
    for path in glob.glob(f"{model_dir}/*.py"):
        text = open(path).read()
        new_text, n = _VERSION_ASSERT.subn("pass", text)
        if n:
            open(path, "w").write(new_text)
            patched += n
    return patched

_IMAGE_BRANCH_ORIGINAL = "if past_key_values is None and torch.any(input_ids == self.tokenizer.img_start_id):"
_IMAGE_BRANCH_PATCHED = (
    "if (past_key_values is None or (hasattr(past_key_values, 'get_seq_length') "
    "and past_key_values.get_seq_length() == 0)) and torch.any(input_ids == self.tokenizer.img_start_id):"
)

def patch_image_cache_check(model_dir):
    """transformers >= 4.45 passes an empty DynamicCache (not None) on the first generation step, and the model
    code only encodes the image when past_key_values is None. Without this patch the image is silently skipped."""
    path = f"{model_dir}/modeling_chexagent.py"
    text = open(path).read()
    if _IMAGE_BRANCH_PATCHED in text:
        return 1
    if _IMAGE_BRANCH_ORIGINAL not in text:
        raise RuntimeError("modeling_chexagent.py does not contain the expected image-branch condition")
    open(path, "w").write(text.replace(_IMAGE_BRANCH_ORIGINAL, _IMAGE_BRANCH_PATCHED))
    return 1

def align_visual_dtype(model, dtype):
    """The visual tower keeps a float32 position-embedding parameter and its encode() casts the pixels to the dtype of the
    first parameter it finds, so once the image branch really runs the pixels are float32 while the SigLIP convolution
    weights are float16 ("Input type (FloatTensor) and weight type (HalfTensor) should be the same"). Give the tower one dtype."""
    towers = [m for m in model.modules() if hasattr(m, "encode") and hasattr(m, "attn_pool")]
    if len(towers) != 1:
        raise RuntimeError(f"expected one visual tower, found {len(towers)}; inspect the model code")
    towers[0].to(dtype=dtype)
    found = {p.dtype for p in towers[0].parameters()}
    if found != {dtype}:
        raise RuntimeError(f"visual tower still has mixed dtypes: {found}")
    return towers[0]

model_dir = snapshot_download(MODEL_NAME, local_dir="CheXagent-2-3b")
print("Patched version checks:", patch_version_asserts(model_dir))
print("Image-branch patch in place:", patch_image_cache_check(model_dir))

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir, device_map="auto", torch_dtype=DTYPE, trust_remote_code=True
)
align_visual_dtype(model, DTYPE)
model.eval()
print("Model loaded; visual tower dtype:", DTYPE)


In [ ]:
GENERAL_PROMPT = "{question}"
LATERALITY_SEVERITY_PROMPT = "{question} Answer concisely based only on what is visible in the image."

def prompt_for_category(question, category):
    """Deliberately minimal (no few-shot examples, no retrieval) -- that's the
    whole point of a *baseline*. Phase 2 is where retrieved context gets added."""
    if category in ("laterality", "severity"):
        return LATERALITY_SEVERITY_PROMPT.format(question=question)
    return GENERAL_PROMPT.format(question=question)


def ask_chexagent(image_path, question, max_new_tokens=128):
    query = tokenizer.from_list_format([{"image": image_path}, {"text": question}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
            max_new_tokens=max_new_tokens,
        )[0]
    return tokenizer.decode(output[input_ids.size(1):-1]).strip()


In [ ]:
# IMAGE CHECK: proves the picture reaches the model. The same file path is used with three different pixel patterns,
# so the text the model reads is identical every time; only the pixels differ. If the first-token scores do not change,
# the image is being ignored and nothing below this cell can be trusted.
import numpy as np
from PIL import Image

def _first_token_logprobs(path):
    query = tokenizer.from_list_format([{"image": path}, {"text": "Describe the image."}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
                             max_new_tokens=1, output_scores=True, return_dict_in_generate=True)
    return torch.log_softmax(out.scores[0][0].float(), dim=-1).cpu(), int(input_ids.size(1))

_rng = np.random.default_rng(0)
_variants = {"black": np.zeros((512, 512, 3), np.uint8), "white": np.full((512, 512, 3), 255, np.uint8),
             "noise": _rng.integers(0, 256, (512, 512, 3), dtype=np.uint8)}
_lp = {}
for _name, _pix in _variants.items():
    Image.fromarray(_pix).save("image_check.png")
    _lp[_name], _n_tokens = _first_token_logprobs("image_check.png")
_diffs = {f"{a} vs {b}": float((_lp[a] - _lp[b]).abs().max()) for a, b in (("black", "white"), ("black", "noise"), ("white", "noise"))}
print("prompt length with image slots:", _n_tokens, "tokens (context limit 2048)")
print("largest change in next-token log-probability when only the pixels change:", {k: round(v, 3) for k, v in _diffs.items()})
if max(_diffs.values()) < 0.01:
    raise RuntimeError("IMAGE CHECK FAILED: changing the pixels does not change the model's output, so the image is not "
                       "reaching the model. Stop and report this.")
print("IMAGE CHECK PASSED: the model's output depends on the pixels.")


## 4. Run inference over the test set

In [ ]:
from tqdm.auto import tqdm

for r in tqdm(test_set):
    prompt = prompt_for_category(r["question"], r["category"])
    r["prediction"] = ask_chexagent(r["image_path"], prompt)

print("Done. Example:")
print(test_set[0]["question"], "->", test_set[0]["prediction"])


## 5. Score predictions

Several automated scores are reported side by side, not blended into one number --
Med-Flamingo's own finding was that automated text-overlap metrics can *disagree sharply*
with what a clinician would call correct, so none of them is the final word (see section 6).

- `exact`: strict normalized equality. Fair for yes/no, harsh for open answers.
- `loose`: substring containment either way.
- `token_f1`: word overlap, gives partial credit to open answers.
- Every accuracy has a **95% confidence interval**. With a few dozen questions per bucket,
  intervals that overlap mean the difference between buckets is not proven.
- **Yes/no block:** sensitivity (correct when the finding IS present) and specificity (correct
  when it is NOT), plus the accuracy of always answering the majority class. A model that says
  "no" to almost everything looks fine on accuracy but misses real findings.

The next cell is the repo's own `baseline/evaluate.py`, inlined unchanged.


In [ ]:
"""Answer scoring for the baseline run.

Med-Flamingo's own finding (clinician ratings disagreeing sharply with
automated text-overlap metrics) is why several automated scores are reported
side by side rather than one. None is a substitute for a human spot-check.

- `exact_match`: strict, normalized string equality. Fair for yes/no, harsh for
  open-ended answers.
- `loose_match`: substring containment either direction after normalization.
- `token_f1`: word-overlap F1 (SQuAD style). Gives partial credit to open
  answers like "Chest tightness, shortness of breath" vs "Chest tightness,
  fatigue", which exact_match scores 0.

Every accuracy is reported with a Wilson 95% confidence interval, because with
a few dozen questions per bucket most differences between buckets are not
statistically meaningful, and the report should say so.

Yes/no questions get extra treatment (sensitivity/specificity and the
"always answer the majority class" baseline): a model that answers "no" to
almost everything scores well when most true answers are "no", but misses
real findings.
"""


import math
import re
from dataclasses import dataclass, field

_YES_SYNONYMS = {"yes", "y", "yeah", "affirmative", "true", "present"}
_NO_SYNONYMS = {"no", "n", "nope", "negative", "false", "absent", "not present"}

_PUNCT_RE = re.compile(r"[^a-z0-9\s]")
_WS_RE = re.compile(r"\s+")


def normalize(text: str) -> str:
    text = text.lower().strip()
    text = _PUNCT_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    if text in _YES_SYNONYMS:
        return "yes"
    if text in _NO_SYNONYMS:
        return "no"
    return text


def exact_match(prediction: str, gold: str) -> bool:
    return normalize(prediction) == normalize(gold)


def loose_match(prediction: str, gold: str) -> bool:
    p, g = normalize(prediction), normalize(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    # Guard against trivially short strings matching almost anything.
    shorter, longer = (p, g) if len(p) <= len(g) else (g, p)
    if len(shorter) < 3:
        return False
    return shorter in longer


def token_f1(prediction: str, gold: str) -> float:
    p_tokens, g_tokens = normalize(prediction).split(), normalize(gold).split()
    if not p_tokens or not g_tokens:
        return float(p_tokens == g_tokens)
    remaining = list(g_tokens)
    common = 0
    for tok in p_tokens:
        if tok in remaining:
            remaining.remove(tok)
            common += 1
    if common == 0:
        return 0.0
    precision, recall = common / len(p_tokens), common / len(g_tokens)
    return 2 * precision * recall / (precision + recall)


def wilson_interval(correct: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (float("nan"), float("nan"))
    p = correct / total
    denom = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denom
    half = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


@dataclass
class Stats:
    total: int = 0
    exact_correct: int = 0
    loose_correct: int = 0
    f1_sum: float = 0.0

    def add(self, em: bool, lm: bool, f1: float) -> None:
        self.total += 1
        self.exact_correct += int(em)
        self.loose_correct += int(lm)
        self.f1_sum += f1

    @property
    def exact_accuracy(self) -> float:
        return self.exact_correct / self.total if self.total else float("nan")

    @property
    def loose_accuracy(self) -> float:
        return self.loose_correct / self.total if self.total else float("nan")

    @property
    def mean_f1(self) -> float:
        return self.f1_sum / self.total if self.total else float("nan")

    @property
    def exact_ci(self) -> tuple[float, float]:
        return wilson_interval(self.exact_correct, self.total)


# Backwards-compatible name used by earlier notebook versions and tests.
CategoryStats = Stats


@dataclass
class YesNoStats:
    """Confusion counts for yes/no questions, treating 'yes' as the positive
    class (i.e. 'there is a finding')."""
    tp: int = 0   # gold yes, predicted yes
    fn: int = 0   # gold yes, predicted not-yes  (missed finding)
    tn: int = 0   # gold no,  predicted no
    fp: int = 0   # gold no,  predicted not-no   (false alarm)

    @property
    def total(self) -> int:
        return self.tp + self.fn + self.tn + self.fp

    @property
    def sensitivity(self) -> float:
        pos = self.tp + self.fn
        return self.tp / pos if pos else float("nan")

    @property
    def specificity(self) -> float:
        neg = self.tn + self.fp
        return self.tn / neg if neg else float("nan")

    @property
    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.total if self.total else float("nan")

    @property
    def majority_baseline(self) -> float:
        """Accuracy of always answering the more common gold class."""
        return max(self.tp + self.fn, self.tn + self.fp) / self.total if self.total else float("nan")

    @property
    def predicted_yes_rate(self) -> float:
        return (self.tp + self.fp) / self.total if self.total else float("nan")

    @property
    def gold_yes_rate(self) -> float:
        return (self.tp + self.fn) / self.total if self.total else float("nan")


@dataclass
class EvaluationReport:
    per_category: dict[str, Stats] = field(default_factory=dict)
    per_source: dict[str, Stats] = field(default_factory=dict)
    per_format: dict[str, Stats] = field(default_factory=dict)
    per_category_format: dict[str, Stats] = field(default_factory=dict)
    per_kind: dict[str, Stats] = field(default_factory=dict)
    overall: Stats = field(default_factory=Stats)
    yes_no: YesNoStats = field(default_factory=YesNoStats)
    yes_no_by_category: dict[str, YesNoStats] = field(default_factory=dict)
    rows: list[dict] = field(default_factory=list)


def _gold_format(rec: dict) -> str:
    if rec.get("answer_format"):
        return rec["answer_format"]
    return "yes/no" if normalize(rec["gold_answer"]) in ("yes", "no") else "open"


def evaluate(records: list[dict]) -> EvaluationReport:
    """Score a list of records.

    Required keys: 'question', 'gold_answer', 'prediction', 'category',
    'source_dataset'. Optional: 'answer_format' ('yes/no'|'open', inferred if
    absent) and 'is_knowledge' (bool). Returns overall and per-bucket stats plus
    per-row scoring so a human can inspect individual disagreements.
    """
    report = EvaluationReport()

    for rec in records:
        em = exact_match(rec["prediction"], rec["gold_answer"])
        lm = loose_match(rec["prediction"], rec["gold_answer"])
        f1 = token_f1(rec["prediction"], rec["gold_answer"])
        fmt = _gold_format(rec)
        kind = "knowledge" if rec.get("is_knowledge") else "perception"

        for bucket, key in (
            (report.per_category, rec["category"]),
            (report.per_source, rec["source_dataset"]),
            (report.per_format, fmt),
            (report.per_category_format, f"{rec['category']} / {fmt}"),
            (report.per_kind, kind),
        ):
            bucket.setdefault(key, Stats()).add(em, lm, f1)
        report.overall.add(em, lm, f1)

        if fmt == "yes/no":
            gold_yes = normalize(rec["gold_answer"]) == "yes"
            pred = normalize(rec["prediction"])
            for yn in (report.yes_no, report.yes_no_by_category.setdefault(rec["category"], YesNoStats())):
                if gold_yes:
                    yn.tp += int(pred == "yes")
                    yn.fn += int(pred != "yes")
                else:
                    yn.tn += int(pred == "no")
                    yn.fp += int(pred != "no")

        report.rows.append({**rec, "answer_format": fmt, "exact_match": em,
                            "loose_match": lm, "token_f1": round(f1, 3)})

    return report


def _fmt_stats(s: Stats) -> str:
    lo, hi = s.exact_ci
    return (f"n={s.total:<4} exact={s.exact_accuracy:6.1%} (95% CI {lo:.0%}-{hi:.0%})"
            f"  loose={s.loose_accuracy:6.1%}  token_f1={s.mean_f1:.2f}")


def format_report(report: EvaluationReport) -> str:
    L = []
    L.append("=== Overall ===")
    L.append(_fmt_stats(report.overall))

    def section(title, bucket):
        L.append("")
        L.append(f"=== {title} ===")
        for key, s in sorted(bucket.items()):
            L.append(f"{key:>24}: {_fmt_stats(s)}")

    section("By question category (laterality / severity / general)", report.per_category)
    section("By answer format (yes/no vs open-ended)", report.per_format)
    section("By category and answer format", report.per_category_format)
    section("By question kind (knowledge = symptoms/treatment/etc; perception = look at the image)", report.per_kind)
    section("By source dataset", report.per_source)

    yn = report.yes_no
    L.append("")
    L.append("=== Yes/no questions: does the model miss findings? ===")
    if yn.total:
        L.append(f"n={yn.total}  accuracy={yn.accuracy:.1%}  "
                 f"always-answer-majority baseline={yn.majority_baseline:.1%}")
        L.append(f"sensitivity (correct on gold 'yes')={yn.sensitivity:.1%}  "
                 f"specificity (correct on gold 'no')={yn.specificity:.1%}")
        L.append(f"gold 'yes' rate={yn.gold_yes_rate:.1%}  model predicted 'yes' rate={yn.predicted_yes_rate:.1%}")
        for cat, c in sorted(report.yes_no_by_category.items()):
            L.append(f"  {cat:>12}: n={c.total:<3} acc={c.accuracy:.0%}  sens={c.sensitivity:.0%}  spec={c.specificity:.0%}")
    else:
        L.append("(no yes/no questions in this run)")
    L.append("")
    L.append("Note: buckets with n < ~50 have wide confidence intervals; treat differences that "
             "the intervals overlap on as unproven.")
    return "\n".join(L)


In [ ]:
report = evaluate(test_set)
test_set = report.rows   # same records, now with exact_match / loose_match / token_f1
print(format_report(report))


## 6. Qualitative spot-check (don't skip this)

Automated metrics can diverge from what a clinician would call correct, and the VQA-RAD chest
filter is a keyword heuristic. So look at the **wrong** answers yourself: is the image really a
chest X-ray? Is the model's answer actually wrong, or just worded differently? Laterality and
open-ended mistakes are shown first.


In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

wrong = [r for r in test_set if not r["exact_match"]]
wrong.sort(key=lambda r: (r["category"] != "laterality", r["answer_format"] != "open"))
to_review = wrong[:12]
print(f"{len(wrong)} wrong answers in total; showing {len(to_review)}")

for r in to_review:
    img = Image.open(r["image_path"]).convert("RGB")
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap="gray")
    plt.axis("off")
    plt.title(f"{r['category']} / {r['answer_format']} / {r['source_dataset']}", fontsize=8)
    plt.show()
    print(f"Q: {r['question']}")
    print(f"Gold: {r['gold_answer']!r}   Pred: {r['prediction']!r}   token_f1={r['token_f1']}")
    print("-" * 60)


## 7. Save results

In [ ]:
import json, csv

with open("baseline_results.jsonl", "w") as f:
    for r in test_set:
        f.write(json.dumps({k: v for k, v in r.items() if k != "image_key"}) + "\n")

fieldnames = ["id", "source_dataset", "source_split", "category", "answer_format", "is_knowledge",
              "image_path", "question", "gold_answer", "prediction", "exact_match", "loose_match",
              "token_f1"]
with open("baseline_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in test_set:
        writer.writerow({k: r.get(k) for k in fieldnames})

print("Saved baseline_results.jsonl and baseline_results.csv")

try:
    from google.colab import files
    files.download("baseline_results.csv")
    files.download("baseline_results.jsonl")
except Exception as exc:
    print("Auto-download unavailable; use the Colab file browser (left sidebar) instead.", exc)
print("Keep these files: the Colab VM is wiped when the session ends.")


## What's next (Phase 2)

Read the report in section 5 with these questions:
1. **Which buckets are lowest, and do their confidence intervals really separate from the others?**
   (the paper's weak spot is fine-grained laterality/severity)
2. **Yes/no block:** is sensitivity far below specificity? That means the model under-calls findings,
   which retrieved text cannot fix because the error is in what the model *sees*.
3. **Knowledge vs perception:** errors on `knowledge` questions (symptoms, treatment, how a finding is
   measured) are what a retrieval layer can help with; perception errors point to confidence scoring
   or targeted fine-tuning instead.

Keep `baseline_results.csv`. Phase 2 results will be scored by the same code and compared row by
row (IDs are stable) against this baseline.
